In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data/processed/listings_clean.csv')
print(f'Loaded: {df.shape[0]:,} × {df.shape[1]}')

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Loaded: 58,736 × 78


### Creating Missing Indicator Flags

In [2]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing': missing, 'Pct': missing_pct})
missing_df = missing_df[missing_df.Missing > 0].sort_values('Pct', ascending=False)

print(f'Columns with missing: {len(missing_df)}/{len(df.columns)}')
print(missing_df.head(35))

Columns with missing: 35/78
                             Missing        Pct
neighborhood_overview          35750  60.865568
neighbourhood                  35749  60.863865
host_neighbourhood             34060  57.988287
host_about                     27752  47.248706
host_location                  16275  27.708731
review_scores_value            12010  20.447426
review_scores_location         12010  20.447426
review_scores_checkin          12010  20.447426
review_scores_accuracy         12009  20.445723
review_scores_communication    12009  20.445723
review_scores_cleanliness      12009  20.445723
reviews_per_month              12007  20.442318
last_review                    12007  20.442318
review_scores_rating           12007  20.442318
first_review                   12007  20.442318
host_response_rate              5087   8.660787
host_response_time              5087   8.660787
host_acceptance_rate            3752   6.387905
host_is_superhost               1330   2.264369
description 

In [3]:
# Features where missingness is informative
missing_flag_features = [
    'bedrooms', 'bathrooms', 'beds',
    'review_scores_rating', 'review_scores_accuracy',
    'review_scores_cleanliness', 'review_scores_location',
    'host_response_rate', 'host_acceptance_rate',
    'neighbourhood','host_neighbourhood',
    'has_availability','host_is_superhost'
]

created = 0
for feat in missing_flag_features:
    if feat in df.columns and df[feat].isnull().sum() > 0:
        df[f'{feat}_missing'] = df[feat].isnull().astype(int)
        created += 1

print(f'Created {created} missing indicator flags')

Created 13 missing indicator flags


### Converting Percentage Strings

In [4]:
# Convert '95%' → 0.95
for col in ['host_response_rate', 'host_acceptance_rate']:
    if col in df.columns and df[col].dtype == 'object':
        df[col] = df[col].str.replace('%', '').astype(float) / 100
        print(f'Converted {col}')

Converted host_response_rate
Converted host_acceptance_rate


### Imputing Missing Values

In [5]:
# Review scores - imputing by property_type
review_cols = [c for c in df.columns if c.startswith('review_scores_') and not c.endswith('_missing')]

if 'property_type' in df.columns:
    for col in review_cols:
        df[col] = df.groupby('property_type')[col].transform(lambda x: x.fillna(x.median()))
        df[col] = df[col].fillna(df[col].median())
else:
    for col in review_cols:
        df[col] = df[col].fillna(df[col].median())

print(f'Imputed {len(review_cols)} review features')

# Host features
for col in ['host_response_rate', 'host_acceptance_rate']:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())
        print(f'Imputed {col}')

# Property features - impute by accommodates
for col in ['bedrooms', 'bathrooms', 'beds']:
    if col in df.columns:
        if 'accommodates' in df.columns:
            df[col] = df.groupby('accommodates')[col].transform(lambda x: x.fillna(x.median()))
        df[col] = df[col].fillna(df[col].median())
        print(f'Imputed {col}')

Imputed 7 review features
Imputed host_response_rate
Imputed host_acceptance_rate
Imputed bedrooms
Imputed bathrooms
Imputed beds


### Creating Pricing Features

In [6]:
# Price per person/bedroom/bed
if 'accommodates' in df.columns:
    df['price_per_person'] = df['price'] / df['accommodates'].replace(0, 1)
if 'bedrooms' in df.columns:
    df['price_per_bedroom'] = df['price'] / df['bedrooms'].replace(0, 1)
if 'beds' in df.columns:
    df['price_per_bed'] = df['price'] / df['beds'].replace(0, 1)

df['price_log'] = np.log1p(df['price'])

print('Created pricing features')

Created pricing features


### Creating Property Efficiency Features

In [7]:
if 'beds' in df.columns and 'bedrooms' in df.columns:
    df['beds_per_bedroom'] = df['beds'] / df['bedrooms'].replace(0, 1)
if 'bathrooms' in df.columns and 'bedrooms' in df.columns:
    df['bathrooms_per_bedroom'] = df['bathrooms'] / df['bedrooms'].replace(0, 1)
if 'accommodates' in df.columns and 'bedrooms' in df.columns:
    df['capacity_per_bedroom'] = df['accommodates'] / df['bedrooms'].replace(0, 1)
if 'room_type' in df.columns:
    df['is_entire_home'] = (df['room_type'] == 'Entire home/apt').astype(int)

print('Created efficiency features')

Created efficiency features


### Creating Review Features

In [8]:
if 'number_of_reviews' in df.columns:
    df['has_reviews'] = (df['number_of_reviews'] > 0).astype(int)

if 'last_review' in df.columns:
    #Missing indicator since this feature rejected MCAR relationship with price
    df['last_review_date'] = pd.to_datetime(df['last_review'], errors='coerce')
    df['has_reviews'] = (~df['last_review_date'].isna()).astype(int)
    ref_date = df['last_review_date'].max()
    df['days_since_last_review'] = (ref_date - df['last_review_date']).dt.days
    df['days_since_last_review'] = df['days_since_last_review'].fillna(9999)
    df['has_recent_review'] = (df['days_since_last_review'] < 90).astype(int)

# Average review score
review_score_cols = [c for c in df.columns if c.startswith('review_scores_') and not c.endswith('_missing')]
if len(review_score_cols) > 0:
    df['avg_review_score'] = df[review_score_cols].mean(axis=1)

print('Created review features')

Created review features


### Creating Host Features

In [9]:
# Converting t/f to binary
for col in ['host_is_superhost', 'host_identity_verified', 'host_has_profile_pic']:
    if col in df.columns and df[col].dtype == 'object':
        df[col] = (df[col] == 't').astype(int)

# Professional host
if 'host_listings_count' in df.columns:
    df['is_professional_host'] = (df['host_listings_count'] > 5).astype(int)

# Host quality score
host_quality_features = ['host_is_superhost', 'host_identity_verified', 
                         'host_has_profile_pic', 'host_response_rate']
existing = [f for f in host_quality_features if f in df.columns]
if len(existing) >= 3:
    df['host_quality_score'] = df[existing].apply(
        lambda x: x / x.max() if x.max() > 0 else x, axis=0
    ).mean(axis=1)

print('Created host features')

Created host features


### Creating Location Features

In [10]:
# Neighborhood features
neighborhood_col = None
if 'neighbourhood_cleansed' in df.columns:
    neighborhood_col = 'neighbourhood_cleansed'
elif 'neighbourhood' in df.columns:
    neighborhood_col = 'neighbourhood'

if neighborhood_col:
    # Median price in neighborhood
    df['neighborhood_median_price'] = df.groupby(neighborhood_col)['price'].transform('median')
    
    # Relative pricing
    df['price_vs_neighborhood'] = df['price'] / df['neighborhood_median_price']
    
    # Competition
    df['neighborhood_listing_count'] = df.groupby(neighborhood_col)[neighborhood_col].transform('count')
    threshold = df['neighborhood_listing_count'].quantile(0.75)
    df['is_high_competition_area'] = (df['neighborhood_listing_count'] >= threshold).astype(int)
    
    print('Created location features')

Created location features


### Creating Availability Features

In [11]:
if 'availability_365' in df.columns:
    df['availability_rate'] = df['availability_365'] / 365
    df['is_highly_available'] = (df['availability_rate'] > 0.5).astype(int)
    
    # Booking proxy
    if 'number_of_reviews' in df.columns:
        df['booking_rate_proxy'] = df['number_of_reviews'] / (df['availability_365'] + 1)

if 'minimum_nights' in df.columns:
    df['is_flexible_booking'] = (df['minimum_nights'] <= 3).astype(int)

print('Created availability features')

Created availability features


### Creating Amenity Features

In [12]:
if 'amenities' in df.columns:
    # Count amenities
    df['amenity_count'] = df['amenities'].apply(
        lambda x: len(eval(x)) if pd.notnull(x) and x != '[]' else 0
    )
    
    # High-value amenities
    for amenity in ['Pool', 'Hot tub', 'Gym']:
        col_name = f'has_{amenity.lower().replace(" ", "_")}'
        df[col_name] = df['amenities'].apply(lambda x: 1 if amenity in str(x) else 0)
    
    # Essential amenities
    essential = ['Wifi', 'Kitchen', 'Heating', 'Air conditioning']
    df['essential_amenity_count'] = df['amenities'].apply(
        lambda x: sum(1 for a in essential if a in str(x))
    )
    
    print('Created amenity features')

Created amenity features


### Encoding Categorical Features

In [13]:
from sklearn.preprocessing import LabelEncoder

categorical_features = df.select_dtypes(include=['object', 'category']).columns
categorical_features = [c for c in categorical_features if c in df.columns]

for cat in categorical_features:
    le = LabelEncoder()
    df[f'{cat}_encoded'] = le.fit_transform(df[cat].fillna('Unknown'))

print(f'Encoded {len(categorical_features)} categorical features')

Encoded 28 categorical features


### Handling Remaining Missing Data

In [14]:
# Dropping text columns that won't be used
text_columns_to_drop = [
    'description', 'neighborhood_overview', 'host_about',
    'picture_url', 'host_thumbnail_url', 'host_picture_url'
]
df = df.drop(columns=[col for col in text_columns_to_drop if col in df.columns])

# Handling remaining categorical with 'Unknown'
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna('Unknown')
        print(f'Filled {col} (categorical) with "Unknown"')

# Dropping last review date since it has been transformed and missing indicators created
df = df.drop(columns = ['last_review_date'])

# Handling remaining numeric with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        if pd.isna(median_val):
            df[col] = df[col].fillna(0)
        else:
            df[col] = df[col].fillna(median_val)
        print(f'Filled {col} (numeric) with median/0')

# Final verification
remaining_nulls = df.isnull().sum().sum()

if remaining_nulls == 0:
    print('Zero nulls remaining')
else:
    print(f'{remaining_nulls} NaNs still remaining')
    print('\nColumns with NaNs:')
    print(df.isnull().sum()[df.isnull().sum() > 0])
    
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

Filled host_name (categorical) with "Unknown"
Filled host_since (categorical) with "Unknown"
Filled host_location (categorical) with "Unknown"
Filled host_response_time (categorical) with "Unknown"
Filled host_neighbourhood (categorical) with "Unknown"
Filled host_verifications (categorical) with "Unknown"
Filled neighbourhood (categorical) with "Unknown"
Filled bathrooms_text (categorical) with "Unknown"
Filled has_availability (categorical) with "Unknown"
Filled first_review (categorical) with "Unknown"
Filled last_review (categorical) with "Unknown"
Filled host_listings_count (numeric) with median/0
Filled host_total_listings_count (numeric) with median/0
Filled reviews_per_month (numeric) with median/0
Zero nulls remaining
Final shape: 58,736 rows × 140 columns


### Save Engineered Data

In [15]:
# Save full dataset
df.to_csv('data/processed/listings_engineered.csv', index=False)
print(f'Saved: listings_engineered.csv ({df.shape})')

# Create feature list for modeling
exclude = ['id', 'host_id', 'price', 'price_log', 'is_anomaly', 'anomaly_score', 'scrape_id', 'last_scraped', 'calendar_last_scraped']
features_for_modeling = [c for c in df.columns if c not in exclude]

df[features_for_modeling].to_csv('data/processed/modeling_features.csv', index=False)

print(f'Features for modeling: {len(features_for_modeling)}')

Saved: listings_engineered.csv ((58736, 140))
Features for modeling: 131


### Summary

In [16]:
print(f'Dataset: {len(df):,} listings')
print(f'Total features: {df.shape[1]}')
print(f'Modeling features: {len(features_for_modeling)}')
print()
print('Created:')
print(f'Missing flags: {created}')
print(f'Pricing features: 4')
print(f'Property features: 4')
print(f'Review features: 6')
print(f'Host features: 2')
print(f'Location features: 4')
print(f'Availability features: 4')
print(f'Amenity features: 5')
print()
print('Next steps:')
print('1. Customer segmentation (K-Means)')
print('2. Demand modeling (RF, NN)')
print('3. Price optimization')
print('4. A/B testing validation')

Dataset: 58,736 listings
Total features: 140
Modeling features: 131

Created:
Missing flags: 13
Pricing features: 4
Property features: 4
Review features: 6
Host features: 2
Location features: 4
Availability features: 4
Amenity features: 5

Next steps:
1. Customer segmentation (K-Means)
2. Demand modeling (RF, NN)
3. Price optimization
4. A/B testing validation
